In [ ]:
"""
Adversarial Search - Minimax Algorithm with Alpha-Beta Pruning
Game: Tic-Tac-Toe
"""

import math

# ─────────────────────────────────────────────
#  Board Representation
# ─────────────────────────────────────────────

def create_board():
    """Return an empty 3x3 board (list of 9 cells)."""
    return [' '] * 9


def print_board(board):
    """Pretty-print the current board state."""
    print("\n")
    for row in range(3):
        cells = board[row * 3: row * 3 + 3]
        print("  " + " | ".join(cells))
        if row < 2:
            print("  " + "-" * 9)
    print()


def get_available_moves(board):
    """Return list of indices where the cell is empty."""
    return [i for i, cell in enumerate(board) if cell == ' ']


def make_move(board, index, player):
    """Place player's mark on the board at given index."""
    new_board = board[:]
    new_board[index] = player
    return new_board


# ─────────────────────────────────────────────
#  Win / Terminal Conditions
# ─────────────────────────────────────────────

WIN_CONDITIONS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),   # rows
    (0, 3, 6), (1, 4, 7), (2, 5, 8),   # columns
    (0, 4, 8), (2, 4, 6)               # diagonals
]


def check_winner(board, player):
    """Return True if the given player has won."""
    return any(
        board[a] == board[b] == board[c] == player
        for a, b, c in WIN_CONDITIONS
    )


def is_terminal(board):
    """Return True if the game is over (win or draw)."""
    return (
        check_winner(board, 'X') or
        check_winner(board, 'O') or
        len(get_available_moves(board)) == 0
    )


def evaluate(board):
    """
    Heuristic / utility function.
    +10  ->  AI ('O') wins
    -10  ->  Human ('X') wins
      0  ->  Draw
    """
    if check_winner(board, 'O'):
        return 10
    if check_winner(board, 'X'):
        return -10
    return 0


# ─────────────────────────────────────────────
#  Minimax with Alpha-Beta Pruning
# ─────────────────────────────────────────────

def minimax(board, depth, is_maximizing, alpha, beta):
    """
    Minimax algorithm with Alpha-Beta Pruning.

    Parameters
    ----------
    board           : current board state
    depth           : current recursion depth (used to prefer faster wins)
    is_maximizing   : True if it's the AI's turn (maximizer)
    alpha           : best score the maximizer can guarantee so far
    beta            : best score the minimizer can guarantee so far

    Returns
    -------
    int : the optimal score for the current player
    """
    score = evaluate(board)

    # ── Terminal states ──────────────────────
    if score == 10:
        return score - depth   # prefer quicker wins
    if score == -10:
        return score + depth   # prefer slower losses
    if len(get_available_moves(board)) == 0:
        return 0               # draw

    moves = get_available_moves(board)

    if is_maximizing:
        # AI ('O') tries to maximise the score
        best = -math.inf
        for move in moves:
            new_board = make_move(board, move, 'O')
            score = minimax(new_board, depth + 1, False, alpha, beta)
            best = max(best, score)
            alpha = max(alpha, best)
            if beta <= alpha:
                break           # beta cut-off (pruning)
        return best
    else:
        # Human ('X') tries to minimise the score
        best = math.inf
        for move in moves:
            new_board = make_move(board, move, 'X')
            score = minimax(new_board, depth + 1, True, alpha, beta)
            best = min(best, score)
            beta = min(beta, best)
            if beta <= alpha:
                break           # alpha cut-off (pruning)
        return best


def get_best_move(board):
    """
    Find and return the best move index for the AI ('O') using Minimax.
    """
    best_score = -math.inf
    best_move  = None

    print("  [AI thinking...]")
    for move in get_available_moves(board):
        new_board  = make_move(board, move, 'O')
        move_score = minimax(new_board, 0, False, -math.inf, math.inf)
        print(f"  Move {move} -> score = {move_score}")
        if move_score > best_score:
            best_score = move_score
            best_move  = move

    print(f"  Best move: {best_move} (score = {best_score})")
    return best_move


# ─────────────────────────────────────────────
#  Game Loop
# ─────────────────────────────────────────────

def play_game():
    """Main game loop: Human (X) vs AI (O)."""
    board = create_board()

    print("=" * 45)
    print("  Tic-Tac-Toe -- Human (X) vs AI (O)")
    print("  Board positions:")
    print("    0 | 1 | 2")
    print("    ---------")
    print("    3 | 4 | 5")
    print("    ---------")
    print("    6 | 7 | 8")
    print("=" * 45)

    while True:
        print_board(board)

        # ── Human's Turn ──────────────────────
        available = get_available_moves(board)
        print(f"  Available positions: {available}")
        while True:
            try:
                human_move = int(input("  Your move (0-8): "))
                if human_move in available:
                    break
                print("  Invalid move. Choose from available positions.")
            except ValueError:
                print("  Please enter a number between 0 and 8.")

        board = make_move(board, human_move, 'X')

        if check_winner(board, 'X'):
            print_board(board)
            print("  Congratulations! You (X) win!")
            break
        if not get_available_moves(board):
            print_board(board)
            print("  It's a Draw!")
            break

        # ── AI's Turn ─────────────────────────
        print("\n  AI is computing its best move...")
        ai_move = get_best_move(board)
        board   = make_move(board, ai_move, 'O')
        print(f"\n  AI played at position: {ai_move}")

        if check_winner(board, 'O'):
            print_board(board)
            print("  AI (O) wins! Better luck next time.")
            break
        if not get_available_moves(board):
            print_board(board)
            print("  It's a Draw!")
            break

    print("\n  Thanks for playing!")


# ─────────────────────────────────────────────
#  Entry Point
# ─────────────────────────────────────────────

if __name__ == "__main__":
    play_game()
